In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/Users/vayunandan/Documents/sentiment-analysis/data/data.csv')

In [3]:
df.head

<bound method NDFrame.head of                                                   tweet language sentiment
0     Lionel Messi, que ha estado vinculado con un t...       es   3 stars
1     This is a guest post by The Joy of Truth. To r...       en   4 stars
2     Nous sommes tous conscients de la popularité d...       fr   5 stars
3     El baño en el sistema de metro de la ciudad de...       es   4 stars
4     "Ich habe dies seit über 20 Jahren getan und i...       de   5 stars
...                                                 ...      ...       ...
4912  \nA former CIA officer and CIA director has pl...       en    1 star
4913  Karen M. Felt, Ph.D. La ricerca è stata condot...       it   4 stars
4914  Mit all der Aufmerksamkeit, die dem Thema Abtr...       de   2 stars
4915  L'élément le plus important dans le processus ...       fr   4 stars
4916  On voit souvent dans les films quelqu'un qui a...       fr   3 stars

[4917 rows x 3 columns]>

In [4]:
df_copy = df.copy()

In [5]:
df_copy.drop('language', axis=1, inplace = True)

In [6]:
df_copy.head

<bound method NDFrame.head of                                                   tweet sentiment
0     Lionel Messi, que ha estado vinculado con un t...   3 stars
1     This is a guest post by The Joy of Truth. To r...   4 stars
2     Nous sommes tous conscients de la popularité d...   5 stars
3     El baño en el sistema de metro de la ciudad de...   4 stars
4     "Ich habe dies seit über 20 Jahren getan und i...   5 stars
...                                                 ...       ...
4912  \nA former CIA officer and CIA director has pl...    1 star
4913  Karen M. Felt, Ph.D. La ricerca è stata condot...   4 stars
4914  Mit all der Aufmerksamkeit, die dem Thema Abtr...   2 stars
4915  L'élément le plus important dans le processus ...   4 stars
4916  On voit souvent dans les films quelqu'un qui a...   3 stars

[4917 rows x 2 columns]>

In [7]:
new_sentiment = {
    "5 stars" : 1,
    "4 stars" : 1,
    "3 stars" : 0,
    "2 stars" : -1,
    "1 star" : -1
}

df_copy['new_sentiment'] = df_copy['sentiment'].map(new_sentiment)

In [8]:
df_copy.head
df_copy = df_copy.drop(columns='sentiment', axis = 1)

In [9]:
df_copy = df_copy.rename(columns={'new_sentiment' : 'labels'})

In [11]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df_copy,
    test_size = 0.2,
    stratify = df_copy['labels']
)

In [13]:
train_df.head(5)

,tweet,labels
1008,One of the worst things about Trump's rise was...,-1
2498,(AAP) India e Irak han sido declarados zonas s...,-1
3911,"\nA couple of years back, I had one of the mos...",1
1915,La última vez que fui al hospital fue en septi...,-1
803,La stagione 2016 di Derek Jeter lo ha visto co...,1


In [14]:
len(train_df)

3933

In [15]:
len(temp_df)

984

In [16]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['labels']
)

In [53]:
import matplotlib as plt
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objects as go

df_counts = train_df['new_sentiment'].value_counts().reset_index()
df_counts.columns = ['sentiment_value', 'counts']

# 2. Map the integer values to strings explicitly
label_map = {-1: 'Negative', 0: 'Neutral', 1: 'Positive'}
df_counts['label'] = df_counts['sentiment_value'].map(label_map)

# 3. Plot using the DataFrame columns
# This ensures color '1' always stays with 'Positive', etc.
fig = px.bar(df_counts, 
             x='label', 
             y='counts',
             color='sentiment_value',
             color_continuous_scale='Viridis', # Or your Dark24 sequence
             title='<b>Sentiments Counts</b>')

fig.update_layout(xaxis_title='Sentiment',
                  yaxis_title='Counts',
                  template='plotly_dark',
                  coloraxis_showscale=False) # Hides the color bar on the right

fig.show()
pyo.plot(fig, filename='Sentiments_Counts.html', auto_open=True)

'Sentiments_Counts.html'

In [17]:
train_pos = train_df[train_df['labels']==-1]
len(train_pos)

1518

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd

# Load your data (assuming you already have train_df and val_df)
# train_df should have columns like 'text' and 'label'
label_map = {-1: 0, 0: 1, 1: 2}
train_df['label'] = train_df['labels'].map(label_map)

# Convert pandas DataFrames to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)  # if you have validation data

# Load model and tokenizer
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)  # adjust num_labels

model.gradient_checkpointing_enable()
# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples['tweet'], padding='max_length', truncation=True, max_length=64)

# Tokenize datasets
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=6,
    gradient_accumulation_steps= 2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    use_mps_device=True  # M1 GPU acceleration
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Train!
trainer.train()

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2abfc57b-7f12-4d85-a260-9963910dbc7e)')' thrown while requesting HEAD https://huggingface.co/xlm-roberta-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].


KeyboardInterrupt: 